In [0]:
# Databricks notebook source
# Studio 4 - Relazione tra vento medio e produzione eolica
#
# Qui imposto:
# - librerie;
# - catalogo;
# - tabella Gold finale;
# - colori usati nel grafico;
# - controllo iniziale sui dati;
# - funzioni comuni per lettura dati, grafici e conversioni.

# Importo le librerie principali.
import pandas as pd
import plotly.graph_objects as go

from plotly.subplots import make_subplots


# Imposto il catalogo e la tabella Gold finale.
catalogo = "progetto_meteo"
tabella_dati_studio = f"{catalogo}.gold.dati_studio"

colore_eolico = "#40DD3B"
colore_vento = "#E67E22"


# Seleziono il catalogo del progetto.
spark.sql(f"USE CATALOG {catalogo}")


# Controllo che la Gold finale sia popolata.
# Se la tabella è vuota, i casi studio non possono essere eseguiti.
righe_gold = spark.table(tabella_dati_studio).count()

if righe_gold == 0:
    raise Exception(
        f"La tabella {tabella_dati_studio} è vuota. "
        "Prima esegui Gold_Dati_Aggregati e Gold_Dati_Studio."
    )

print(f"Tabella pronta: {tabella_dati_studio}")
print(f"Righe disponibili: {righe_gold}")


# Leggo una query direttamente da Databricks.
# Dentro Databricks il notebook può usare Spark senza credenziali locali.
def leggi_databricks(query):

    return spark.sql(query).toPandas()


# Mostro i grafici Plotly dentro Databricks.
# displayHTML rende il grafico più stabile nei notebook Databricks.
def mostra_grafico(fig):

    displayHTML(
        fig.to_html(
            include_plotlyjs=True,
            full_html=False,
            config={"responsive": True}
        )
    )


# Converto colonne pandas in numerico quando serve.
# Questo evita problemi di visualizzazione con Plotly.
def converti_colonne_numeriche(df, colonne):

    for colonna in colonne:
        if colonna in df.columns:
            df[colonna] = pd.to_numeric(df[colonna], errors="coerce")

    return df

In [0]:
# Studio 4 - Relazione tra vento medio e produzione eolica
#
# Questo studio confronta due grandezze collegate:
# - la velocità media del vento;
# - la produzione eolica.
#
# L'analisi viene fatta su base mensile e separata per macroarea.
# Per ogni area vengono mostrati:
# - barre della produzione eolica mensile;
# - linea del vento medio mensile;
# - correlazione tra vento medio e produzione eolica.
#
# L'obiettivo è capire se, nei dati aggregati del progetto,
# i mesi con vento medio più alto coincidono anche con una maggiore produzione eolica.
#
# Questo confronto è utile perché l'eolico è una fonte rinnovabile fortemente dipendente
# dalle condizioni meteorologiche. Guardare insieme vento e produzione permette di leggere
# meglio il comportamento territoriale delle macroaree.
#
# In ambito gestionale può aiutare a valutare dove la produzione eolica segue meglio
# l'andamento meteorologico e dove invece potrebbero incidere altri fattori,
# come distribuzione degli impianti, capacità installata o caratteristiche del territorio.
#
# In ambito economico o di investimento può essere usato come base per analisi preliminari
# sulla stabilità della produzione eolica e sulla sensibilità delle macroaree alla variabilità del vento.


# Imposto l'anno da analizzare.
anno_studio = 2025


# Leggo vento medio mensile e produzione eolica mensile dalla Gold finale.
query_vento_eolico = f"""
SELECT
    Area,
    date_format(to_date(Data, 'dd/MM/yyyy'), 'yyyy-MM') AS Mese,
    CAST(ROUND(AVG(Velocita_Vento), 2) AS DOUBLE) AS Vento_Medio,
    CAST(ROUND(SUM(Eolico) / 1000000, 2) AS DOUBLE) AS Eolico_TWh
FROM {tabella_dati_studio}
WHERE year(to_date(Data, 'dd/MM/yyyy')) = {anno_studio}
  AND Velocita_Vento IS NOT NULL
GROUP BY
    Area,
    date_format(to_date(Data, 'dd/MM/yyyy'), 'yyyy-MM')
ORDER BY
    Area,
    Mese
"""

df_vento_eolico = leggi_databricks(query_vento_eolico)


# Converto le colonne numeriche.
df_vento_eolico = converti_colonne_numeriche(
    df_vento_eolico,
    [
        "Vento_Medio",
        "Eolico_TWh"
    ]
)


# Controllo i dati letti.
display(df_vento_eolico)

print("Righe lette:", len(df_vento_eolico))


# Calcolo la correlazione mensile vento/eolico per area.
# Evito groupby.apply perché può dare errori con alcune versioni Pandas.
correlazioni = []

for area in df_vento_eolico["Area"].dropna().unique():

    df_area = df_vento_eolico[df_vento_eolico["Area"] == area]

    if len(df_area) >= 2:
        valore_correlazione = df_area["Vento_Medio"].corr(df_area["Eolico_TWh"])
    else:
        valore_correlazione = None

    if pd.notna(valore_correlazione):
        valore_correlazione = round(float(valore_correlazione), 2)
    else:
        valore_correlazione = None

    correlazioni.append({
        "Area": area,
        "Correlazione": valore_correlazione
    })


# Creo il dataframe delle correlazioni.
df_correlazioni = pd.DataFrame(correlazioni)

display(df_correlazioni)


# Creo una mappa con le correlazioni.
mappa_correlazioni = dict(
    zip(
        df_correlazioni["Area"],
        df_correlazioni["Correlazione"]
    )
)


# Imposto l'ordine delle aree.
aree = [
    "Nord",
    "Centro",
    "Sud",
    "Isole"
]


# Creo i titoli dei riquadri.
titoli = [
    f"{area} | corr: {mappa_correlazioni.get(area)}"
    for area in aree
]


# Creo un grafico con un riquadro per ogni area.
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=titoli,
    specs=[
        [{"secondary_y": True}, {"secondary_y": True}],
        [{"secondary_y": True}, {"secondary_y": True}]
    ]
)


# Aggiungo barre e linea per ogni area.
for indice, area in enumerate(aree):

    riga = indice // 2 + 1
    colonna = indice % 2 + 1

    dati_area = df_vento_eolico[df_vento_eolico["Area"] == area]

    # Produzione eolica mensile.
    fig.add_trace(
        go.Bar(
            x=dati_area["Mese"],
            y=dati_area["Eolico_TWh"],
            name="Eolico",
            marker_color=colore_eolico,
            showlegend=(indice == 0),
            hovertemplate="Mese: %{x}<br>Eolico: %{y:.2f} TWh<extra></extra>"
        ),
        row=riga,
        col=colonna,
        secondary_y=False
    )

    # Vento medio mensile.
    fig.add_trace(
        go.Scatter(
            x=dati_area["Mese"],
            y=dati_area["Vento_Medio"],
            name="Vento medio",
            mode="lines+markers",
            line=dict(color=colore_vento, width=3),
            marker=dict(size=7),
            showlegend=(indice == 0),
            hovertemplate="Mese: %{x}<br>Vento medio: %{y:.2f} km/h<extra></extra>"
        ),
        row=riga,
        col=colonna,
        secondary_y=True
    )


# Sistemo il layout.
fig.update_layout(
    title=f"Relazione tra vento medio e produzione eolica per macroarea - {anno_studio}",
    title_x=0.5,
    height=800,
    hovermode="x unified",
    legend_title="Indicatore"
)

fig.update_yaxes(title_text="Eolico (TWh)", secondary_y=False)
fig.update_yaxes(title_text="Vento medio (km/h)", secondary_y=True)

fig.update_xaxes(tickangle=-45)


# Mostro il grafico.
mostra_grafico(fig)